In [10]:
import csv

# 1) Define JobSeeker and JobOffer classes to hold our data -----------------

class JobSeeker:
    def __init__(self, skills, experience, salary, location,
                 job_interest, sector, education_level, job_id):
        self.skills = skills
        self.experience = experience
        self.salary = salary
        self.location = location
        self.job_interest = job_interest
        self.sector = sector
        self.education_level = education_level
        self.job_id = job_id

    def __repr__(self):
        return (f"JobSeeker(id={self.job_id}, skills={self.skills}, "
                f"exp={self.experience}, sal={self.salary}, loc={self.location})")


class JobOffer:
    def __init__(self, required_skills, min_experience,
                 salary_range, location, sector, education_level, offer_id):
        self.required_skills = required_skills
        self.min_experience = min_experience
        self.salary_range = salary_range  # tuple (min, max)
        self.location = location
        self.sector = sector
        self.education_level = education_level
        self.offer_id = offer_id

    def __repr__(self):
        return (f"JobOffer(id={self.offer_id}, req={self.required_skills}, "
                f"exp={self.min_experience}, sal_range={self.salary_range}, loc={self.location})")


# 2) Load data from CSV files ------------------------------------------------

def load_job_seekers(filename):
    seekers = []
    try:
        with open(filename, mode='r', newline='') as f:
            reader = csv.DictReader(f)
            for row in reader:
                skills = [s.strip() for s in row['skills'].split(',') if s.strip()]
                exp = int(row['experience'])
                sal = int(row['salary'])
                loc = row['location']
                job_interest = row.get('job_interest', '')
                sector = row.get('sector', '')
                education = row.get('education_level', '')
                job_id = row.get('job_id', '')
                seekers.append(JobSeeker(
                    skills, exp, sal, loc,
                    job_interest, sector, education, job_id
                ))
    except FileNotFoundError:
        print(f"Error: file {filename} not found.")
    except Exception as e:
        print(f"Error reading {filename}: {e}")
    return seekers


def load_job_offers(filename):
    offers = []
    try:
        with open(filename, mode='r', newline='') as f:
            reader = csv.DictReader(f)
            for i, row in enumerate(reader):
                req_skills = [s.strip() for s in row['required_skills'].split(',') if s.strip()]
                min_exp = int(row['min_experience'])

                # Flexible salary range parsing
                salary_raw = row['salary_range'].replace('(', '').replace(')', '')
                if '-' in salary_raw:
                    salary_parts = salary_raw.split('-')
                elif ',' in salary_raw:
                    salary_parts = salary_raw.split(',')
                else:
                    raise ValueError(f"Unexpected salary_range format: {salary_raw}")
                smin = int(salary_parts[0].strip())
                smax = int(salary_parts[1].strip())

                location = row['location']
                sector = row.get('sector', '')
                education = row.get('education_level', '')
                offer_id = row.get('offer_id', f"OF{i}")
                offers.append(JobOffer(
                    req_skills, min_exp, (smin, smax), location, sector, education, offer_id
                ))
    except FileNotFoundError:
        print(f"Error: file {filename} not found.")
    except Exception as e:
        print(f"Error reading {filename}: {e}")
    return offers



# 3) Heuristic function computing a “matching score” for a seeker+offer -----

def improved_heuristic(seeker: JobSeeker, offer: JobOffer) -> float:
    missing = set(offer.required_skills) - set(seeker.skills)
    skill_pen = len(missing) * 10
    exp_pen = max(0, offer.min_experience - seeker.experience) * 5
    if seeker.salary < offer.salary_range[0]:
        sal_pen = (offer.salary_range[0] - seeker.salary) / 1000
    elif seeker.salary > offer.salary_range[1]:
        sal_pen = (seeker.salary - offer.salary_range[1]) / 1000
    else:
        sal_pen = 0
    loc_pen = 0 if seeker.location.lower() == offer.location.lower() else 15
    return -(skill_pen + exp_pen + sal_pen + loc_pen)


# 4) Build the score matrix -------------------------------------------------

def build_score_matrix(seekers, offers):
    matrix = []
    for s in seekers:
        row = []
        for o in offers:
            row.append(improved_heuristic(s, o))
        matrix.append(row)
    return matrix


# 5) Greedy matching over the matrix ----------------------------------------

def greedy_match_from_matrix(matrix, seekers, offers):
    assigned = []
    available_rows = set(range(len(seekers)))
    available_cols = set(range(len(offers)))
    for _ in range(min(len(seekers), len(offers))):
        best = None
        best_score = float('-inf')
        for i in available_rows:
            for j in available_cols:
                if matrix[i][j] > best_score:
                    best_score = matrix[i][j]
                    best = (i, j)
        if not best:
            break
        i_best, j_best = best
        assigned.append((offers[j_best], seekers[i_best], best_score))
        available_rows.remove(i_best)
        available_cols.remove(j_best)
    return assigned


# 6) Main driver -------------------------------------------------------------

def main():
    seekers = load_job_seekers('job_seekers_with_ids.csv')
    offers = load_job_offers('job_offers_ascii_clean.csv')
    if not seekers or not offers:
        print('Ensure both CSV files exist and have correct headers.')
        return
    matrix = build_score_matrix(seekers, offers)
    assignment = greedy_match_from_matrix(matrix, seekers, offers)
    total = sum(score for _, _, score in assignment)
    avg = total / len(assignment) if assignment else 0
    print(f"Assigned {len(assignment)} pairs.")
    print(f"Total matching score: {total:.2f}")
    print(f"Average score per match: {avg:.2f}\n")
    print('All assignments:')
    for offer, seeker, score in assignment:
        print(f"  {offer.offer_id} ↔ {seeker.job_id}  score={score:.2f}")
    print("\nNote: greedy may not yield the global optimum sum.")


if __name__ == '__main__':
    main()


Assigned 1000 pairs.
Total matching score: -39395.37
Average score per match: -39.40

All assignments:
  OF302 ↔ 10000  score=-10.00
  OF352 ↔ 10001  score=-10.00
  OF513 ↔ 10003  score=-10.00
  OF85 ↔ 10004  score=-10.00
  OF24 ↔ 10008  score=-10.00
  OF47 ↔ 10009  score=-10.00
  OF403 ↔ 10012  score=-10.00
  OF521 ↔ 10013  score=-10.00
  OF120 ↔ 10017  score=-10.00
  OF232 ↔ 10021  score=-10.00
  OF192 ↔ 10024  score=-10.00
  OF686 ↔ 10028  score=-10.00
  OF269 ↔ 10029  score=-10.00
  OF981 ↔ 10032  score=-10.00
  OF69 ↔ 10034  score=-10.00
  OF548 ↔ 10035  score=-10.00
  OF131 ↔ 10038  score=-10.00
  OF7 ↔ 10041  score=-10.00
  OF127 ↔ 10042  score=-10.00
  OF994 ↔ 10044  score=-10.00
  OF346 ↔ 10045  score=-10.00
  OF209 ↔ 10049  score=-10.00
  OF428 ↔ 10059  score=-10.00
  OF34 ↔ 10060  score=-10.00
  OF350 ↔ 10071  score=-10.00
  OF826 ↔ 10076  score=-10.00
  OF549 ↔ 10078  score=-10.00
  OF893 ↔ 10097  score=-10.00
  OF270 ↔ 10100  score=-10.00
  OF446 ↔ 10102  score=-10.00
  OF